# Developer Sentiment on Open-Source AI Models (2023–2026)

A quantitative corpus study analyzing GitHub issue discourse across five major open-source AI/ML repositories to trace sentiment trends toward seven open-weight model families: Llama, Qwen, Gemma, DeepSeek, Phi, Kimi, and GLM, between 2023 and 2026.

**Data source:** GitHub REST Search API  
**Repositories:** `pytorch/pytorch`, `huggingface/transformers`, `tensorflow/tensorflow`, `ollama/ollama`, `AUTOMATIC1111/stable-diffusion-webui`  
**Method:** Lexicon-based sentiment scoring (VADER) + keyword-based model tagging

---

In [ ]:
import requests
import pandas as pd
import time
from getpass import getpass

# Enter my GitHub token securely  
token = getpass("Enter your GitHub token: ")
headers = {"Authorization": f"token {token}"}

# Quick sanity check that auth works
resp = requests.get("https://api.github.com/user", headers=headers, timeout=10)
print(resp.status_code, resp.json().get("login"))

In [3]:
repos = [
    "pytorch/pytorch",
    "huggingface/transformers",
    "tensorflow/tensorflow",
    "ollama/ollama",
    "AUTOMATIC1111/stable-diffusion-webui"
]

model_config = {
    "Llama":    {"keywords": ["llama 2", "llama2", "llama-2", "llama 3", "llama3", "llama-3",
                               "llama 4", "llama4", "llama-4"]},
    "Phi":      {"keywords": ["phi-2", "phi2", "phi-3", "phi3", "phi-4", "phi4"]},
    "Qwen":     {"keywords": ["qwen", "qwen2", "qwen2.5", "qwen3", "qwen-vl", "qwen-coder"]},
    "Gemma":    {"keywords": ["gemma 2", "gemma2", "gemma-2", "gemma 3", "gemma3", "gemma-3",
                               "gemma 4", "gemma4", "gemma-4"]},
    "DeepSeek": {"keywords": ["deepseek", "deepseek-v2", "deepseek-v3", "deepseek-r1",
                               "deepseek v3.2", "deepseek-v4"]},
    "Kimi":     {"keywords": ["kimi k2", "kimi-k2", "kimi k2.5", "kimi k2.6", "kimi k2.7"]},
    "GLM":      {"keywords": ["glm-5", "glm5", "glm-5.1", "glm-5.2"]},
    "Open Source (general)": {"keywords": ["open source", "open-source", "open weight", "open-weight"]}
}

years = [
    ("2023-01-01", "2023-12-31"),
    ("2024-01-01", "2024-12-31"),
    ("2025-01-01", "2025-12-31"),
    ("2026-01-01", "2026-07-01")
]

In [5]:
all_issues = []
error_log = []

for repo in repos:
    for start, end in years:
        for page in range(1, 11):  # up to 1000/repo/year 
            url = "https://api.github.com/search/issues"
            params = {
                "q": f"repo:{repo} type:issue created:{start}..{end}",
                "per_page": 100,
                "page": page,
                "sort": "created",
                "order": "asc"
            }
            resp = requests.get(url, headers=headers, params=params, timeout=10)

            if resp.status_code != 200:
                print(f"ERROR {resp.status_code} for {repo} {start}-{end} page {page}")
                error_log.append((repo, start, end, page, resp.status_code, resp.text))
                time.sleep(5)
                continue

            items = resp.json().get("items", [])
            if not items:
                break

            all_issues.extend(items)
            time.sleep(2)

    print(f"Done: {repo} — running total: {len(all_issues)}")

print("\nFINAL TOTAL:", len(all_issues))

Done: pytorch/pytorch — running total: 4000
Done: huggingface/transformers — running total: 7751
Done: tensorflow/tensorflow — running total: 11236
Done: ollama/ollama — running total: 15236
Done: AUTOMATIC1111/stable-diffusion-webui — running total: 17307

FINAL TOTAL: 17307


In [6]:
df = pd.DataFrame(all_issues)
print(df.shape)

# Save raw data right away so I never have to re-run the API calls
df.to_csv("github_issues_raw.csv", index=False)
print("Saved raw data.")

(17307, 35)
Saved raw data.


In [7]:
df["repo_source"] = df["html_url"].apply(lambda x: "/".join(x.split("/")[3:5]))
df["created_at"] = pd.to_datetime(df["created_at"])
df["year"] = df["created_at"].dt.year
df["full_text"] = (df["title"].fillna("") + " " + df["body"].fillna("")).str.lower()

print(df["year"].value_counts().sort_index())
print(df["repo_source"].value_counts())

year
2023    5000
2024    4856
2025    4103
2026    3348
Name: count, dtype: int64
repo_source
pytorch/pytorch                         4000
ollama/ollama                           4000
huggingface/transformers                3751
tensorflow/tensorflow                   3485
AUTOMATIC1111/stable-diffusion-webui    2071
Name: count, dtype: int64


In [8]:
def find_matches(text):
    matches = []
    for model, cfg in model_config.items():
        if any(kw.lower() in text for kw in cfg["keywords"]):
            matches.append(model)
    return matches

df["matched_models"] = df["full_text"].apply(find_matches)
df["num_matches"] = df["matched_models"].apply(len)

print(df["num_matches"].value_counts())
print(df[df["num_matches"] > 0]["matched_models"].explode().value_counts())

num_matches
0    15043
1     2017
2      203
3       31
4        8
5        5
Name: count, dtype: int64
matched_models
Qwen                     785
Llama                    738
DeepSeek                 399
Gemma                    288
Open Source (general)    231
Phi                       99
GLM                       17
Kimi                      16
Name: count, dtype: int64


In [13]:
df.to_csv("github_issues_tagged.csv", index=False)
print("Saved:", df.shape)

Saved: (17307, 40)
